## Local Inference on GPU
Model page: https://huggingface.co/lxyuan/distilbert-base-multilingual-cased-sentiments-student

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/lxyuan/distilbert-base-multilingual-cased-sentiments-student)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [1]:
!pip install -U transformers
!pip install datasets
!pip install pandas
!pip install scikit-learn
!pip install openpyxl
!pip install torch

  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
Using cached transformers-5.5.0-py3-none-any.whl (10.2 MB)


In [2]:
import pandas as pd

df = pd.read_excel("DataGabungan.xlsx")

df.head()

,TOPIK,Username Posting owner,Link post IG,Username komentar,Isi komentar,Sentimen
0,Tentara Nasional Indonesia (TNI) berniat menel...,tempodotco,https://www.instagram.com/p/DLPF7LDs29b/,halidi.iy,Guaaaaa,-1
1,Tentara Nasional Indonesia (TNI) berniat menel...,tempodotco,https://www.instagram.com/p/DLPF7LDs29b/,sagi.saputra_,"Lucu, lalu setelah tau dalangnya, mau diapain??",-1
2,Tentara Nasional Indonesia (TNI) berniat menel...,tempodotco,https://www.instagram.com/p/DLPF7LDs29b/,lzrd.nzr,Maksudnya ga boleh nolak?,-1
3,Tentara Nasional Indonesia (TNI) berniat menel...,tempodotco,https://www.instagram.com/p/DLPF7LDs29b/,febrisheget,"Saya dalangnya pak, sini dah ngopi. saya cerit...",0
4,Tentara Nasional Indonesia (TNI) berniat menel...,tempodotco,https://www.instagram.com/p/DLPF7LDs29b/,setyaryantio,Yang lain siap siap ww3 yang disini masih ngur...,-1


In [3]:
df = df[["Isi komentar", "Sentimen"]]

df = df.rename(columns={
    "Isi komentar": "text",
    "Sentimen": "label"
})

df.head()


,text,label
0,Guaaaaa,-1
1,"Lucu, lalu setelah tau dalangnya, mau diapain??",-1
2,Maksudnya ga boleh nolak?,-1
3,"Saya dalangnya pak, sini dah ngopi. saya cerit...",0
4,Yang lain siap siap ww3 yang disini masih ngur...,-1


In [4]:
df["label"].value_counts()

,count
label,
0,7424
-1,6224
1,3309


In [5]:
# Gunakan .replace() lebih aman daripada .map() untuk menghindari nilai NaN
df["label"] = df["label"].replace({
    -1: 0,
    0: 1,
    1: 2
})

In [6]:
df["label"].value_counts()

,count
label,
1,7424
0,6224
2,3309


In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(
    "lxyuan/distilbert-base-multilingual-cased-sentiments-student"
)

model = AutoModelForSequenceClassification.from_pretrained(
    "lxyuan/distilbert-base-multilingual-cased-sentiments-student",
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [8]:
df = df.dropna(subset=["text"])
df = df[df["text"].str.strip() != ""]



In [9]:
df["text"] = df["text"].astype(str)

df.head()

,text,label
0,Guaaaaa,0
1,"Lucu, lalu setelah tau dalangnya, mau diapain??",0
2,Maksudnya ga boleh nolak?,0
3,"Saya dalangnya pak, sini dah ngopi. saya cerit...",1
4,Yang lain siap siap ww3 yang disini masih ngur...,0


In [10]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42
)

In [11]:
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=128
)

In [12]:
import torch

class SentimentDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = SentimentDataset(train_encodings, train_labels)
test_dataset = SentimentDataset(test_encodings, test_labels)

In [13]:
print(f"Total data: {len(df)}")
print(f"Train: {len(train_texts)}")
print(f"Test: {len(test_texts)}")

Total data: 16957
Train: 13565
Test: 3392


In [14]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=6,          # tambah epoch
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    warmup_steps=500,             # tambahkan warmup
    weight_decay=0.01,            # tambahkan regularisasi
    load_best_model_at_end=True,  # simpan model terbaik
)

In [15]:
# Tambahkan cell ini sebelum WeightedTrainer
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=train_labels
)
print(class_weights)

[0.90324943 0.77016976 1.68216766]


In [16]:
import torch
from transformers import Trainer

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        weights = torch.tensor(class_weights, dtype=torch.float).to(logits.device)
        loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [17]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,1.072217,0.876207
2,0.754817,0.817659
3,0.573546,0.861669
4,0.442967,0.989004
5,0.334353,1.204842
6,0.269653,1.359498


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=5088, training_loss=0.5597717799480606, metrics={'train_runtime': 1160.2409, 'train_samples_per_second': 70.149, 'train_steps_per_second': 4.385, 'total_flos': 2695428462435840.0, 'train_loss': 0.5597717799480606, 'epoch': 6.0})

In [18]:
from sklearn.metrics import classification_report
import numpy as np

predictions = trainer.predict(test_dataset)

preds = np.argmax(predictions.predictions, axis=1)

print(classification_report(test_labels, preds))

              precision    recall  f1-score   support

           0       0.66      0.63      0.65      1218
           1       0.73      0.68      0.70      1553
           2       0.52      0.66      0.59       621

    accuracy                           0.66      3392
   macro avg       0.64      0.66      0.65      3392
weighted avg       0.67      0.66      0.66      3392

